### Attention Mechanisms — Self-Attention, Causal Self-Attention, Multi-Head Attention

Built up incrementally: simplified self-attention -> trainable self-attention -> causal masking -> multiple heads. Follows the progression in Sebastian Raschka's *Build a Large Language Model (From Scratch)*, Ch. 3.

#### 1. Simplified Self-Attention

In [1]:
# STEP 0
import torch
import torch.nn as nn

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)
print(inputs.shape) # With 6 Tokens of each 3 dimensions.

# ---- STEP 1 ------ Attention Scores
query_2 = inputs[1] # second input token as query
attn_scores_2 = torch.empty(inputs.shape[0]) # same shape as input sequence
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query_2)
    # concise way to multiply two vectors element wise.
    # measure of similarity on how close two vectors.
    # determine the extend each element "attends to" any other element
print(attn_scores_2)

# ---- STEP 2 ----- Normalize Attention Scores.
# To use Softamx, better at managing extreme values &
# offer more favourable gradient  properties.
# the output is not negative & interepretable as probabilities
# OPTION A
attn_weights_temp_2 = attn_scores_2 / attn_scores_2.sum()
print("Attention Wieghts", attn_weights_temp_2)
print("Sum of Attention Weights", attn_weights_temp_2.sum()
      )
# OPTION B
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_naive_2 = softmax_naive(attn_scores_2)
print("Attention Wights with Softmax", attn_weights_naive_2)
print("Sum of Attention Weights", attn_weights_naive_2.sum())

# OPTION C
# Softmax naive may have underflow or overflow issues, (num stability)
# Advisable to use PyTorch's Softmax
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention Weights with PyTorch", attn_weights_2)
print("Sum of Attention Weights", attn_weights_2.sum())

# STEP 3 - Calculating Context Vectors
context_vect_2 = torch.zeros(query_2.shape) # second input token
for i, x_i in enumerate(inputs):
    context_vect_2 += attn_weights_2[i] * x_i
print("Context Vector", context_vect_2)

torch.Size([6, 3])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention Wieghts tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of Attention Weights tensor(1.0000)
Attention Wights with Softmax tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of Attention Weights tensor(1.)
Attention Weights with PyTorch tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of Attention Weights tensor(1.)
Context Vector tensor([0.4419, 0.6515, 0.5683])


#### Self Attention For All Token

In [2]:
# SLOWER FOR LOOP
# attn_scores = torch.empty(6, 6)
# for i, x_i in enumerate(inputs):
#     for j, x_j in enumerate(inputs):
#         attn_scores[i, j] = torch.dot(x_i, x_j)
# print("Attention Scores", attn_scores)

# STEP 1 - ATENTION SCORES
attn_scores = inputs @ inputs.T # Matric Multiplications
print(attn_scores)

# STEP 2 - NORMALIZE ATTENTION SCORES - ATTENTION WEIGHTS
attn_weights = torch.softmax(attn_scores, dim=-1)
# apply normalization along last dimension
# it will normalize along columns, so that values in each row sum upto 1
print(attn_weights)

# STEP 3 - CONTEXT VECTORS COMPUTED
context_vecs = attn_weights @ inputs
print("Context Vectors", context_vecs)
# Each row contains three dimensional context vectors

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
Context Vectors tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


#### Self Attention with Trainable Weights


In [3]:
# Trainable weights are crucial so that model can learn
# to produce good context vectors
x_2 = inputs[1] # Second element
d_in = inputs.shape[1] # The embeddings, d=3
d_out = 2 # output embedding, d = 2
# In GPT MOdels, input & Output are same.

torch.manual_seed(123)
# Why Query, Key & Values ?
# -- Search (Query in DB), Key is like a DB Indexing, Searching
# -- Values is the actual content as in key-valu pair in db
# Matrices used to prpject embedded input tokens.
# Would use requires_grad=True, to update matrices during training
W_q = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_k = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_v = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_q # (1, 3) @ (3, 2)
# key_2 = x_2 @ W_k
# value_2 = x_2 @ W_v

# need to obtain all keys & values
keys = inputs @ W_k
values = inputs @ W_v
# Projected to (6, 2) i.e. on to 2 dimensional embedding space.
print("Keys Shape", keys.shape)
print("Values Shape", values.shape)

# Attention Score
attn_scores_2 = query_2 @ keys.T # all attention score for given query
print(attn_scores_2)

# Attention Wieghts
# By dividing them by square root of the embedding dimension of keys
# large dot product - small gradients during backpropogation, due to softmax
# As dot product increases, softmax function behaves like a step function

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)


Keys Shape torch.Size([6, 2])
Values Shape torch.Size([6, 2])
tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.3809])
tensor([0.2854, 0.4081])


#### A compact self-attention class

In [4]:
class SelfAttention_v1(nn.Module): # a class derived from nn.Module
    def __init__(self, d_in, d_out):
        super().__init__()
        # intializes trainable weights, tranforming input d_in to d_out
        self.W_q = nn.Parameter(torch.rand(d_in, d_out))
        self.W_k = nn.Parameter(torch.rand(d_in, d_out))
        self.W_v = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # transform input data into Queries, Keys, values
        queries = x @ self.W_q
        keys = x @ self.W_k
        values = x @ self.W_v

        attn_scores = queries @ keys.T # (6, 2) @ (2, 6) --> (6, 6)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values # (6, 6) @ (6, 2) --> (6, 2)
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(3, 2)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


#### Self Attention class using PyTorch's Linear Layers


In [5]:
# Using nn.Linear has optimized weight initialization scheme,
# leading better model training  (effective & Stable)

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_q(x)
        keys = self.W_k(x)
        values = self.W_v(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(3, 2)
print(sa_v2(inputs))
    

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


#### Comparing SelfAttention_v1 and SelfAttention_v2

Transfer the weight matrices from a SelfAttention_v2 object to a Self-Attention_v1, such that both objects then produce the same results. Your task is to correctly assign the weights from an instance of SelfAttention_v2 to an instance of SelfAttention_v1. To do this, you need to understand the relationship between the weights in both versions.

In [6]:
sa_v2.W_k.weight.data.T
sa_v1.W_k.data

tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])

#### 2. Causal Attention -

Only considers previous and current inputs in a sequnce when processing any given token when computing attention score, While Self attention allows all tokens in an input sequence to be processed at once.

#### Why Causal Attention ? -

#### Steps

Attention Scores  -> Apply Softmax -> Attention Weights(Normalized) ->
 -> Mask with 0's above Diagonal -> Masked Attention Scores (Unnormalized) ->
 -> Normalize rows -> Masked Attention Weights (Normalized)

Reuses the `SelfAttention_v2` (`sa_v2`) instance defined above.

##### 2.1 Applying a Causal Attention Mask

Intially it might appear that mask attention weights and renormalize might still have influence on the current token beacuse thier values are still part of softmax calculations, But since we are re-normalizing masked attention weights, we are essentialy doing is recalculating softmax over the smaller subset (Elegance of Softmax,) Therefor No Information Leakage

In [7]:
# 1. Reuses the query and key weight matrices of the SelfAttention_v2 object
queries = sa_v2.W_q(inputs)
keys = sa_v2.W_k(inputs)
attention_scores = queries @ keys.T
attention_weights = torch.softmax(
    attention_scores / keys.shape[-1] ** 0.5, dim=-1
)
print(attention_weights)
#context_vectors = attention_weights @ sa_v2.W_v(inputs)

# 2. Pytorch's tril function to create a simple mask
context_length =  attention_scores.shape[0]
simple_mask = torch.tril(torch.ones(context_length, context_length))
#print(simple_mask)

# To 0's the attention weights values above diagonal
masked_simple = attention_weights * simple_mask # Multiply
#print(masked_simple)

# 3. Renormalize attention weights to sum up to 1 again in each row
rows_sum = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / rows_sum
#print(masked_simple_norm)

# 4. A more efficient trick
# Softmax converts its input into a probability dist. large negative -> 0
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attention_scores.masked_fill(mask.bool(), -torch.inf) # large negative value

attention_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=-1)
print(attention_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


#### 2.2 Masking additonal attention weights with Dropout

Randomly selected hidden layer unit are ignored during trainig. Prevents overfitting such that it does not become over reliant on any specific hidden layer units.

Dropout in GPT is applied at two times - After calculating attention wieghts or after multiplying attention weights with value vectors

When we apply dropuout to attention weights, Half of the elements in attention wieght matrix is set to 0. To compensate for the reduction in elements, the values of remaining element is scaled  by factor of 1 / 0.5 = 2, to maintain overall balance of attention wieghts, ensurng average influence of attention mechanism remain consistent during both training and inference stages.


In [8]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

print(dropout(attention_weights))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])
tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5085, 0.4936, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3906, 0.0000],
        [0.3249, 0.3418, 0.0000, 0.3308, 0.3249, 0.3363]],
       grad_fn=<MulBackward0>)


#### 2.3 Implementing a compact class



In [9]:
# 2 inputs with 6 token each. each tokem has dimension = 3
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [10]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out,
                 context_length, dropout=0.5, qkv_bias=False):
        super().__init__()
        self.W_q = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length),
                        diagonal=1)
        )
        # buffers are automatically moved to appropiate device, along with our model
        # Hence no need to manually ensure that tensor on same device as model params

    def forward(self, x):
        b, num_tokens, d_in = x.shape # Keeping the batch at dimension  0
        queries = self.W_q(x)
        keys = self.W_k(x)
        values = self.W_v(x)

        attention_scores = queries @ keys.transpose(1, 2) # Transposing dim 1 and 2
        attention_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )# in PyTorch, Operation with trailing _ is performed in-place.
        # Avoiding unnecessary memory copies

        attention_weights = torch.softmax(
            attention_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        attention_weights = self.dropout(attention_weights)
        context_vec = attention_weights @ values

        return context_vec

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(3, 2, context_length, 0.0)
context_vec = ca(batch)
print("context_vecs.shape", context_vec.shape)

context_vecs.shape torch.Size([2, 6, 2])


##### 3. Why do we have muliple attention heads ?

To attend to information from different representation subspaces at differen positions. Module computes several attention in parallel, each with it own learn projection.

Token embedding is identicial for the same sequence for all heads, for each head, model applies a different linear transformation to the embedding to produce Q, K, V. Each head computes attention using Q, K, V as each head sees the input differently. The output are concatenated and mixed, allowing the model to combine information.

##### Q. If all projection matrices start randomly and see the same input, why don't all attention heads learn the same thing ?

W_Q, W_K, W_V - Start with random values. Do not necessarily converge to same solution. Updated during training independently, During BackPropagation, gradient for each head's parameters depend on - head's own output, loss function, interaction with other heads. Hence each head to receive different gradient updates.

- Random Intialization -- Different staring points
- Independent Weights -- Unique learning paths
- Gradient Updates -- Driven by head-specific outputs
- Loss Optimization -- Encourages diversity for better results

Reuses the `CausalAttention` class, `batch`, and `context_length` defined above.


#### Stacking multiple single-head attention


In [11]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False, num_heads=2):
        super().__init__()
        self.heads = torch.nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)
            ])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)
d_in, d_out = batch.shape[-1], 2
context_length = batch.shape[1]

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout=0.0)
print(mha(batch))

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)


#### Multi-Head Attention with weight splits

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # Shape : (d_in, d_out)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 4

mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context vec shape:", context_vecs.shape)

tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)
context vec shape: torch.Size([2, 6, 4])
